# Merge 2018-2019 Floors 3, 4, 5, and 7

This notebook is a focused version of the merge workflow. It only merges the raw CSV files for floors 3, 4, 5, and 7 across 2018 and 2019.

Scope:

- Years: `2018`, `2019`
- Floors: `3`, `4`, `5`, `7`
- Input folder: `Raw Data/`
- Output folder: `outputs/`
- Main output: `combined_wide_2018_2019_floors_3_4_5_7.csv.gz`

The merge is vertical concatenation. It keeps `year`, `floor`, and `source_file` columns so every row can be traced back to its original CSV file.


## 1. Import libraries and locate project folders


In [1]:
from pathlib import Path
from typing import Optional
import re

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_NAME = "Artificial Intelligence for Smart-Building Log Analysis"
YEARS = [2018, 2019]
FLOORS = [3, 4, 5, 7]


def find_project_dir(start: Optional[Path] = None) -> Path:
    """Find the project folder from the current notebook location or parent folders."""
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if path.name == PROJECT_NAME and (path / "Raw Data").exists():
            return path
        candidate = path / PROJECT_NAME
        if (candidate / "Raw Data").exists():
            return candidate
    raise FileNotFoundError(f"Could not find '{PROJECT_NAME}/Raw Data'.")


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "Raw Data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_DIR, DATA_DIR, OUTPUT_DIR


(PosixPath('/Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis'),
 PosixPath('/Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/Raw Data'),
 PosixPath('/Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs'))

## 2. Select the required 8 source CSV files


In [2]:
source_files = [DATA_DIR / f"{year}Floor{floor}.csv" for year in YEARS for floor in FLOORS]
missing_files = [path for path in source_files if not path.exists()]

if missing_files:
    raise FileNotFoundError("Missing required CSV files:\n" + "\n".join(str(path) for path in missing_files))

source_file_table = pd.DataFrame({
    "year": [int(re.search(r"(20\d{2})", path.name).group(1)) for path in source_files],
    "floor": [int(re.search(r"Floor(\d+)", path.name).group(1)) for path in source_files],
    "file_name": [path.name for path in source_files],
    "file_path": [str(path) for path in source_files],
})

source_file_table


,year,floor,file_name,file_path
0,2018,3,2018Floor3.csv,/Users/yonglijiang/Documents/Adelaide Universi...
1,2018,4,2018Floor4.csv,/Users/yonglijiang/Documents/Adelaide Universi...
2,2018,5,2018Floor5.csv,/Users/yonglijiang/Documents/Adelaide Universi...
3,2018,7,2018Floor7.csv,/Users/yonglijiang/Documents/Adelaide Universi...
4,2019,3,2019Floor3.csv,/Users/yonglijiang/Documents/Adelaide Universi...
5,2019,4,2019Floor4.csv,/Users/yonglijiang/Documents/Adelaide Universi...
6,2019,5,2019Floor5.csv,/Users/yonglijiang/Documents/Adelaide Universi...
7,2019,7,2019Floor7.csv,/Users/yonglijiang/Documents/Adelaide Universi...


## 3. Check column counts and field consistency

Floors 3, 4, 5, and 7 should use the same 30 fields. This check prevents accidental merging by position when field names do not match.


In [3]:
headers = {}
for path in source_files:
    headers[path.name] = list(pd.read_csv(path, nrows=0).columns)

reference_file = source_files[0].name
reference_columns = headers[reference_file]

column_check = pd.DataFrame({
    "file_name": list(headers.keys()),
    "column_count": [len(cols) for cols in headers.values()],
    "same_columns_and_order_as_reference": [cols == reference_columns for cols in headers.values()],
})

COLUMN_CHECK_PATH = OUTPUT_DIR / "column_check_2018_2019_floors_3_4_5_7.csv"
column_check.to_csv(COLUMN_CHECK_PATH, index=False)

print("Reference file:", reference_file)
print("Reference column count:", len(reference_columns))
print("Column check saved to:", COLUMN_CHECK_PATH)

if not column_check["same_columns_and_order_as_reference"].all():
    mismatched = column_check.loc[~column_check["same_columns_and_order_as_reference"], "file_name"].tolist()
    raise ValueError(f"Column mismatch found in: {mismatched}")

column_check


Reference file: 2018Floor3.csv
Reference column count: 30
Column check saved to: /Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs/column_check_2018_2019_floors_3_4_5_7.csv


,file_name,column_count,same_columns_and_order_as_reference
0,2018Floor3.csv,30,True
1,2018Floor4.csv,30,True
2,2018Floor5.csv,30,True
3,2018Floor7.csv,30,True
4,2019Floor3.csv,30,True
5,2019Floor4.csv,30,True
6,2019Floor5.csv,30,True
7,2019Floor7.csv,30,True


## 4. Show the 30 columns used in the merge


In [4]:
pd.DataFrame({"column_index": range(1, len(reference_columns) + 1), "column_name": reference_columns})


,column_index,column_name
0,1,Date
1,2,z1_AC1(kW)
2,3,z1_AC2(kW)
3,4,z1_AC3(kW)
4,5,z1_AC4(kW)
5,6,z1_Light(kW)
6,7,z1_Plug(kW)
7,8,z1_S1(degC)
8,9,z1_S1(RH%)
9,10,z1_S1(lux)


## 5. Count rows before merging


In [5]:
row_counts = []
for path in source_files:
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        rows = sum(1 for _ in handle) - 1
    year = int(re.search(r"(20\d{2})", path.name).group(1))
    floor = int(re.search(r"Floor(\d+)", path.name).group(1))
    row_counts.append({"year": year, "floor": floor, "file_name": path.name, "row_count": rows})

row_count_table = pd.DataFrame(row_counts)
ROW_COUNT_PATH = OUTPUT_DIR / "row_counts_2018_2019_floors_3_4_5_7.csv"
row_count_table.to_csv(ROW_COUNT_PATH, index=False)

print("Expected merged rows:", f"{row_count_table['row_count'].sum():,}")
print("Row count summary saved to:", ROW_COUNT_PATH)

row_count_table


Expected merged rows: 3,162,240
Row count summary saved to: /Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs/row_counts_2018_2019_floors_3_4_5_7.csv


,year,floor,file_name,row_count
0,2018,3,2018Floor3.csv,264960
1,2018,4,2018Floor4.csv,264960
2,2018,5,2018Floor5.csv,264960
3,2018,7,2018Floor7.csv,264960
4,2019,3,2019Floor3.csv,525600
5,2019,4,2019Floor4.csv,525600
6,2019,5,2019Floor5.csv,525600
7,2019,7,2019Floor7.csv,525600


## 6. Merge the selected CSV files

This section reads each selected raw file, standardizes the `Date` column to datetime, adds traceability columns, and vertically concatenates the tables.


In [6]:
merged_parts = []

for path in source_files:
    year = int(re.search(r"(20\d{2})", path.name).group(1))
    floor = int(re.search(r"Floor(\d+)", path.name).group(1))

    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df.insert(0, "source_file", path.name)
    df.insert(0, "floor", floor)
    df.insert(0, "year", year)
    merged_parts.append(df)

combined = pd.concat(merged_parts, ignore_index=True)
combined = combined.sort_values(["year", "floor", "Date"]).reset_index(drop=True)

combined.shape


(3162240, 33)

## 7. Validate the merged table


In [7]:
validation_summary = {
    "rows": len(combined),
    "columns": combined.shape[1],
    "expected_rows_from_sources": int(row_count_table["row_count"].sum()),
    "invalid_datetime_rows": int(combined["Date"].isna().sum()),
    "duplicate_year_floor_date_rows": int(combined.duplicated(["year", "floor", "Date"]).sum()),
    "min_date": combined["Date"].min(),
    "max_date": combined["Date"].max(),
}

validation_summary_df = pd.DataFrame([validation_summary])
validation_summary_df


,rows,columns,expected_rows_from_sources,invalid_datetime_rows,duplicate_year_floor_date_rows,min_date,max_date
0,3162240,33,3162240,0,0,2018-07-01,2019-12-31 23:59:00


## 8. Save the merged output


In [8]:
OUTPUT_FILE = OUTPUT_DIR / "combined_wide_2018_2019_floors_3_4_5_7.csv.gz"
PREVIEW_FILE = OUTPUT_DIR / "combined_wide_2018_2019_floors_3_4_5_7_preview.csv"
SUMMARY_FILE = OUTPUT_DIR / "merge_summary_2018_2019_floors_3_4_5_7.csv"

combined.to_csv(OUTPUT_FILE, index=False, compression="gzip")
combined.head(1000).to_csv(PREVIEW_FILE, index=False)
validation_summary_df.to_csv(SUMMARY_FILE, index=False)

print("Merged output saved to:", OUTPUT_FILE)
print("Preview saved to:", PREVIEW_FILE)
print("Merge summary saved to:", SUMMARY_FILE)
print("Rows:", f"{len(combined):,}")
print("Columns:", combined.shape[1])


Merged output saved to: /Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs/combined_wide_2018_2019_floors_3_4_5_7.csv.gz
Preview saved to: /Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs/combined_wide_2018_2019_floors_3_4_5_7_preview.csv
Merge summary saved to: /Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs/merge_summary_2018_2019_floors_3_4_5_7.csv
Rows: 3,162,240
Columns: 33


## 9. Quick review


In [9]:
display(combined.head())
display(combined.groupby(["year", "floor"]).size().reset_index(name="rows"))
combined.shape

,year,floor,source_file,Date,z1_AC1(kW),z1_AC2(kW),z1_AC3(kW),z1_AC4(kW),z1_Light(kW),z1_Plug(kW),z1_S1(degC),z1_S1(RH%),z1_S1(lux),z2_AC1(kW),z2_Light(kW),z2_Plug(kW),z2_S1(degC),z2_S1(RH%),z2_S1(lux),z3_Light(kW),z3_Plug(kW),z4_AC1(kW),z4_Light(kW),z4_Plug(kW),z4_S1(degC),z4_S1(RH%),z4_S1(lux),z5_AC1(kW),z5_Light(kW),z5_Plug(kW),z5_S1(degC),z5_S1(RH%),z5_S1(lux)
0,2018,3,2018Floor3.csv,2018-07-01 00:00:00,0.0,0.0,0.0,0.07,0.0,0.03,28.44,60.67,0.0,0.0,0.0,0.03,27.67,63.37,0.0,0.01,0.26,0.0,0.0,0.02,27.69,65.66,0.0,0.0,0.0,0.02,26.88,71.16,0.0
1,2018,3,2018Floor3.csv,2018-07-01 00:01:00,0.0,0.0,0.0,0.07,0.0,0.03,28.44,60.67,0.0,0.0,0.0,0.03,27.67,63.37,0.0,0.02,0.37,0.0,0.0,0.02,27.69,65.65,0.0,0.0,0.0,0.02,26.88,71.16,0.0
2,2018,3,2018Floor3.csv,2018-07-01 00:02:00,0.0,0.0,0.0,0.07,0.0,0.03,28.45,60.67,0.0,0.0,0.0,0.03,27.67,63.37,0.0,0.01,0.32,0.0,0.0,0.02,27.69,65.63,0.0,0.0,0.0,0.02,26.88,71.18,0.0
3,2018,3,2018Floor3.csv,2018-07-01 00:03:00,0.0,0.0,0.0,0.07,0.0,0.03,28.45,60.66,0.0,0.0,0.0,0.03,27.67,63.36,0.0,0.01,0.26,0.0,0.0,0.02,NaN,NaN,NaN,0.0,0.0,0.02,26.88,71.19,0.0
4,2018,3,2018Floor3.csv,2018-07-01 00:04:00,0.0,0.0,0.0,0.07,0.0,0.03,28.45,60.66,0.0,0.0,0.0,0.03,27.67,63.36,0.0,0.02,0.37,0.0,0.0,0.02,NaN,NaN,NaN,0.0,0.0,0.02,26.88,71.21,0.0


,year,floor,rows
0,2018,3,264960
1,2018,4,264960
2,2018,5,264960
3,2018,7,264960
4,2019,3,525600
5,2019,4,525600
6,2019,5,525600
7,2019,7,525600


(3162240, 33)

## 10. Data cleaning workflow

The cleaning workflow is split into separate cells. Each cleaning step is wrapped in a function and returns an audit table so the notebook clearly shows what was changed.

The workflow uses the merged table named `combined` from the previous section.


In [10]:
# Cleaning configuration
RESAMPLE_INTERVAL = "15min"
EXPECTED_MINUTES_PER_WINDOW = 15

POWER_MIN_VALUE = 0
TEMPERATURE_MIN_C = 10
TEMPERATURE_MAX_C = 45
HUMIDITY_MIN_PCT = 0
HUMIDITY_MAX_PCT = 100
LUX_MIN_VALUE = 0

MEASUREMENT_COLUMNS = [col for col in combined.columns if col.startswith("z")]
POWER_COLUMNS = [col for col in MEASUREMENT_COLUMNS if col.endswith("(kW)")]
ZONE_IDS = sorted({int(match.group(1)) for col in MEASUREMENT_COLUMNS if (match := re.match(r"z(\d+)_", col))})
VALUE_COLUMNS = ["ac_power_kw", "light_power_kw", "plug_power_kw", "total_power_kw", "temperature_c", "humidity_pct", "lux"]
AVAILABILITY_COLUMNS = ["has_ac_meter", "has_light_meter", "has_plug_meter", "has_environment_sensor"]

print("Measurement columns:", len(MEASUREMENT_COLUMNS))
print("Power columns:", len(POWER_COLUMNS))
print("Zones:", ZONE_IDS)


Measurement columns: 29
Power columns: 17
Zones: [1, 2, 3, 4, 5]


## 11. Step 01: standardize timestamp

What this step does:

- Converts `Date` into a unified datetime column named `timestamp`.
- Counts invalid timestamps.
- Removes rows whose timestamp cannot be parsed.
- Returns how many rows were removed.


In [11]:
def clean_step_01_standardize_timestamp(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    before_rows = len(df)
    out = df.copy()
    out["timestamp"] = pd.to_datetime(out["Date"], errors="coerce")

    invalid_timestamp_rows = int(out["timestamp"].isna().sum())
    out = out.dropna(subset=["timestamp"]).reset_index(drop=True)

    audit = pd.DataFrame([{
        "step": "01_standardize_timestamp",
        "action": "Convert Date to timestamp and remove invalid timestamps",
        "rows_before": before_rows,
        "invalid_timestamp_rows": invalid_timestamp_rows,
        "rows_removed": before_rows - len(out),
        "rows_after": len(out),
        "min_timestamp": out["timestamp"].min(),
        "max_timestamp": out["timestamp"].max(),
    }])
    return out, audit

clean_step_01_df, audit_01_timestamp = clean_step_01_standardize_timestamp(combined)
display(audit_01_timestamp)


,step,action,rows_before,invalid_timestamp_rows,rows_removed,rows_after,min_timestamp,max_timestamp
0,01_standardize_timestamp,Convert Date to timestamp and remove invalid t...,3162240,0,0,3162240,2018-07-01,2019-12-31 23:59:00


## 12. Step 02: sort records

What this step does:

- Sorts the data by `floor`, `timestamp`, `year`, and `source_file`.
- Keeps the row count unchanged.
- Returns whether the output order matches the intended sorting keys.


In [12]:
def clean_step_02_sort_records(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    before_rows = len(df)
    sort_keys = ["floor", "timestamp", "year", "source_file"]
    out = df.sort_values(sort_keys).reset_index(drop=True)

    sorted_check = out[sort_keys].equals(out[sort_keys].sort_values(sort_keys).reset_index(drop=True))

    audit = pd.DataFrame([{
        "step": "02_sort_records",
        "action": "Sort by floor, timestamp, year, and source_file",
        "sort_keys": " -> ".join(sort_keys),
        "rows_before": before_rows,
        "rows_removed": 0,
        "rows_after": len(out),
        "is_sorted": bool(sorted_check),
    }])
    return out, audit

clean_step_02_df, audit_02_sort = clean_step_02_sort_records(clean_step_01_df)
display(audit_02_sort)


,step,action,sort_keys,rows_before,rows_removed,rows_after,is_sorted
0,02_sort_records,"Sort by floor, timestamp, year, and source_file",floor -> timestamp -> year -> source_file,3162240,0,3162240,True


## 13. Step 03: remove duplicate records

What this step does:

- Checks duplicate records using `year + floor + timestamp` as the key.
- Removes duplicate rows and keeps the first record.
- Returns how many duplicate rows were found and how many were removed.


In [13]:
def clean_step_03_remove_duplicates(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    before_rows = len(df)
    duplicate_keys = ["year", "floor", "timestamp"]
    duplicate_rows_found = int(df.duplicated(duplicate_keys).sum())

    out = df.drop_duplicates(duplicate_keys, keep="first").reset_index(drop=True)
    duplicate_rows_after = int(out.duplicated(duplicate_keys).sum())

    audit = pd.DataFrame([{
        "step": "03_remove_duplicates",
        "action": "Remove duplicate year-floor-timestamp records",
        "duplicate_keys": " + ".join(duplicate_keys),
        "rows_before": before_rows,
        "duplicate_rows_found": duplicate_rows_found,
        "duplicate_rows_removed": before_rows - len(out),
        "duplicate_rows_after": duplicate_rows_after,
        "rows_after": len(out),
    }])
    return out, audit

clean_step_03_df, audit_03_duplicates = clean_step_03_remove_duplicates(clean_step_02_df)
display(audit_03_duplicates)


,step,action,duplicate_keys,rows_before,duplicate_rows_found,duplicate_rows_removed,duplicate_rows_after,rows_after
0,03_remove_duplicates,Remove duplicate year-floor-timestamp records,year + floor + timestamp,3162240,0,0,0,3162240


## 14. Step 04: standardize numeric measurement columns

What this step does:

- Converts all sensor and meter columns to numeric values.
- Keeps units consistent with the original field names: power in kW, temperature in degC, humidity in RH%, and illuminance in lux.
- Counts values that became missing because they could not be parsed as numbers.


In [14]:
def clean_step_04_standardize_numeric_columns(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = df.copy()
    audit_rows = []

    for col in MEASUREMENT_COLUMNS:
        missing_before = int(out[col].isna().sum())
        converted = pd.to_numeric(out[col], errors="coerce")
        missing_after = int(converted.isna().sum())
        newly_coerced_to_missing = max(0, missing_after - missing_before)
        out[col] = converted

        audit_rows.append({
            "step": "04_standardize_numeric_columns",
            "action": "Convert measurement column to numeric",
            "column": col,
            "unit": "kW" if col.endswith("(kW)") else "degC" if col.endswith("(degC)") else "RH%" if col.endswith("(RH%)") else "lux" if col.endswith("(lux)") else "unknown",
            "missing_before": missing_before,
            "missing_after": missing_after,
            "newly_coerced_to_missing": newly_coerced_to_missing,
        })

    audit = pd.DataFrame(audit_rows)
    return out, audit

clean_step_04_df, audit_04_numeric = clean_step_04_standardize_numeric_columns(clean_step_03_df)
display(audit_04_numeric.head(10))
print("New values coerced to missing:", int(audit_04_numeric["newly_coerced_to_missing"].sum()))


,step,action,column,unit,missing_before,missing_after,newly_coerced_to_missing
0,04_standardize_numeric_columns,Convert measurement column to numeric,z1_AC1(kW),kW,258111,258111,0
1,04_standardize_numeric_columns,Convert measurement column to numeric,z1_AC2(kW),kW,258111,258111,0
2,04_standardize_numeric_columns,Convert measurement column to numeric,z1_AC3(kW),kW,258111,258111,0
3,04_standardize_numeric_columns,Convert measurement column to numeric,z1_AC4(kW),kW,211180,211180,0
4,04_standardize_numeric_columns,Convert measurement column to numeric,z1_Light(kW),kW,153882,153882,0
5,04_standardize_numeric_columns,Convert measurement column to numeric,z1_Plug(kW),kW,102211,102211,0
6,04_standardize_numeric_columns,Convert measurement column to numeric,z1_S1(degC),degC,1151289,1151289,0
7,04_standardize_numeric_columns,Convert measurement column to numeric,z1_S1(RH%),RH%,1151286,1151286,0
8,04_standardize_numeric_columns,Convert measurement column to numeric,z1_S1(lux),lux,1151347,1151347,0
9,04_standardize_numeric_columns,Convert measurement column to numeric,z2_AC1(kW),kW,211098,211098,0


New values coerced to missing: 0


## 15. Step 05: check impossible values

What this step does:

- Checks negative power values.
- Checks temperature outside `10-45` degC.
- Checks humidity outside `0-100` RH%.
- Checks negative lux values.
- Converts impossible values to missing values.
- Returns how many impossible values were found by column and rule.


In [15]:
def clean_step_05_check_impossible_values(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = df.copy()
    audit_rows = []

    rules = []
    for col in MEASUREMENT_COLUMNS:
        if col.endswith("(kW)"):
            rules.append((col, "power_kw < 0", out[col] < POWER_MIN_VALUE))
        elif col.endswith("(degC)"):
            rules.append((col, "temperature_c outside 10-45", (out[col] < TEMPERATURE_MIN_C) | (out[col] > TEMPERATURE_MAX_C)))
        elif col.endswith("(RH%)"):
            rules.append((col, "humidity_pct outside 0-100", (out[col] < HUMIDITY_MIN_PCT) | (out[col] > HUMIDITY_MAX_PCT)))
        elif col.endswith("(lux)"):
            rules.append((col, "lux < 0", out[col] < LUX_MIN_VALUE))

    for col, rule_name, invalid_mask in rules:
        invalid_count = int(invalid_mask.sum())
        if invalid_count:
            out.loc[invalid_mask, col] = pd.NA
        audit_rows.append({
            "step": "05_check_impossible_values",
            "action": "Convert impossible values to missing",
            "column": col,
            "invalid_rule": rule_name,
            "invalid_count": invalid_count,
            "values_changed_to_missing": invalid_count,
        })

    audit = pd.DataFrame(audit_rows)
    return out, audit

clean_step_05_df, audit_05_invalid_values = clean_step_05_check_impossible_values(clean_step_04_df)
display(audit_05_invalid_values.query("invalid_count > 0"))
print("Total impossible values found:", int(audit_05_invalid_values["invalid_count"].sum()))


,step,action,column,invalid_rule,invalid_count,values_changed_to_missing


Total impossible values found: 0


## 16. Step 06: convert wide table to zone-level table

What this step does:

- Converts the wide floor-level table into one row per `timestamp + floor + zone`.
- Aggregates multiple AC meters within a zone into `ac_power_kw`.
- Keeps lighting, plug, temperature, humidity, and lux as standardized fields.
- Adds meter/sensor availability flags.
- Preserves real `0` values and keeps missing values as `NaN`.


In [16]:
def clean_step_06_convert_to_zone_level(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    zone_frames = []
    audit_rows = []

    for (year, floor, source_file), group in df.groupby(["year", "floor", "source_file"], sort=True):
        group = group.sort_values("timestamp").reset_index(drop=True)

        for zone in ZONE_IDS:
            prefix = f"z{zone}_"
            ac_cols = [col for col in group.columns if re.fullmatch(rf"{prefix}AC\d+\(kW\)", col)]
            light_col = f"{prefix}Light(kW)"
            plug_col = f"{prefix}Plug(kW)"
            temp_col = f"{prefix}S1(degC)"
            humidity_col = f"{prefix}S1(RH%)"
            lux_col = f"{prefix}S1(lux)"
            missing_numeric = pd.Series(pd.NA, index=group.index, dtype="Float64")

            zone_df = pd.DataFrame({
                "timestamp": group["timestamp"],
                "year": year,
                "floor": floor,
                "zone": zone,
                "source_file": source_file,
                "has_ac_meter": bool(ac_cols),
                "has_light_meter": light_col in group.columns,
                "has_plug_meter": plug_col in group.columns,
                "has_environment_sensor": all(col in group.columns for col in [temp_col, humidity_col, lux_col]),
            })
            zone_df["ac_power_kw"] = group[ac_cols].sum(axis=1, min_count=1) if ac_cols else missing_numeric.copy()
            zone_df["light_power_kw"] = group[light_col] if light_col in group.columns else missing_numeric.copy()
            zone_df["plug_power_kw"] = group[plug_col] if plug_col in group.columns else missing_numeric.copy()
            zone_df["temperature_c"] = group[temp_col] if temp_col in group.columns else missing_numeric.copy()
            zone_df["humidity_pct"] = group[humidity_col] if humidity_col in group.columns else missing_numeric.copy()
            zone_df["lux"] = group[lux_col] if lux_col in group.columns else missing_numeric.copy()
            zone_df["total_power_kw"] = zone_df[["ac_power_kw", "light_power_kw", "plug_power_kw"]].sum(axis=1, min_count=1)

            zone_frames.append(zone_df)
            audit_rows.append({
                "step": "06_convert_to_zone_level",
                "action": "Convert wide floor-level data to timestamp-floor-zone rows",
                "year": year,
                "floor": floor,
                "zone": zone,
                "source_file": source_file,
                "source_rows": len(group),
                "zone_rows_created": len(zone_df),
                "ac_meter_count": len(ac_cols),
                "has_light_meter": light_col in group.columns,
                "has_plug_meter": plug_col in group.columns,
                "has_environment_sensor": all(col in group.columns for col in [temp_col, humidity_col, lux_col]),
            })

    out = pd.concat(zone_frames, ignore_index=True)
    audit = pd.DataFrame(audit_rows)
    return out, audit

zone_level_minute_df, audit_06_zone_level = clean_step_06_convert_to_zone_level(clean_step_05_df)
display(audit_06_zone_level.head(10))
print("Zone-level minute rows created:", f"{len(zone_level_minute_df):,}")


/var/folders/p4/f4s_7smj75v4r3f2mdqbl2j40000gn/T/ipykernel_17186/3886857628.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(zone_frames, ignore_index=True)


,step,action,year,floor,zone,source_file,source_rows,zone_rows_created,ac_meter_count,has_light_meter,has_plug_meter,has_environment_sensor
0,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,3,1,2018Floor3.csv,264960,264960,4,True,True,True
1,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,3,2,2018Floor3.csv,264960,264960,1,True,True,True
2,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,3,3,2018Floor3.csv,264960,264960,0,True,True,False
3,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,3,4,2018Floor3.csv,264960,264960,1,True,True,True
4,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,3,5,2018Floor3.csv,264960,264960,1,True,True,True
5,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,4,1,2018Floor4.csv,264960,264960,4,True,True,True
6,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,4,2,2018Floor4.csv,264960,264960,1,True,True,True
7,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,4,3,2018Floor4.csv,264960,264960,0,True,True,False
8,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,4,4,2018Floor4.csv,264960,264960,1,True,True,True
9,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,4,5,2018Floor4.csv,264960,264960,1,True,True,True


Zone-level minute rows created: 15,811,200


## 17. Step 07: sort zone-level records

What this step does:

- Sorts the modelling table by `floor -> zone -> timestamp`.
- Keeps row count unchanged.
- Returns whether the final order matches the expected modelling order.


In [17]:
def clean_step_07_sort_zone_records(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    before_rows = len(df)
    sort_keys = ["floor", "zone", "timestamp"]
    out = df.sort_values(sort_keys).reset_index(drop=True)
    sorted_check = out[sort_keys].equals(out[sort_keys].sort_values(sort_keys).reset_index(drop=True))

    audit = pd.DataFrame([{
        "step": "07_sort_zone_records",
        "action": "Sort zone-level records by floor, zone, and timestamp",
        "sort_keys": " -> ".join(sort_keys),
        "rows_before": before_rows,
        "rows_removed": 0,
        "rows_after": len(out),
        "is_sorted": bool(sorted_check),
    }])
    return out, audit

zone_level_minute_sorted_df, audit_07_zone_sort = clean_step_07_sort_zone_records(zone_level_minute_df)
display(audit_07_zone_sort)


,step,action,sort_keys,rows_before,rows_removed,rows_after,is_sorted
0,07_sort_zone_records,"Sort zone-level records by floor, zone, and ti...",floor -> zone -> timestamp,15811200,0,15811200,True


## 18. Step 08: resample to 15-minute intervals

What this step does:

- Resamples each `year + floor + zone` group to 15-minute intervals.
- Calculates mean, min, max, and valid-count features for each measurement.
- Counts missing time windows and incomplete 15-minute windows.
- Does not fill missing measurement values with `0`.


In [18]:
def clean_step_08_resample_15min(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    resampled_parts = []
    audit_rows = []

    for (year, floor, zone), group in df.groupby(["year", "floor", "zone"], sort=True):
        group = group.sort_values("timestamp").set_index("timestamp")
        start = group.index.min().floor(RESAMPLE_INTERVAL)
        end = group.index.max().floor(RESAMPLE_INTERVAL)
        full_index = pd.date_range(start=start, end=end, freq=RESAMPLE_INTERVAL)

        value_mean = group[VALUE_COLUMNS].resample(RESAMPLE_INTERVAL).mean().reindex(full_index)
        value_min = group[VALUE_COLUMNS].resample(RESAMPLE_INTERVAL).min().reindex(full_index).add_suffix("_min")
        value_max = group[VALUE_COLUMNS].resample(RESAMPLE_INTERVAL).max().reindex(full_index).add_suffix("_max")
        valid_counts = group[VALUE_COLUMNS].resample(RESAMPLE_INTERVAL).count().reindex(full_index, fill_value=0).add_suffix("_valid_count")
        observed_minutes = group["source_file"].resample(RESAMPLE_INTERVAL).count().reindex(full_index, fill_value=0)

        out = pd.concat([value_mean, value_min, value_max, valid_counts], axis=1)
        out.insert(0, "timestamp", full_index)
        out.insert(1, "year", year)
        out.insert(2, "floor", floor)
        out.insert(3, "zone", zone)
        out["observed_minutes"] = observed_minutes.to_numpy()
        out["missing_time_flag"] = out["observed_minutes"] == 0
        out["incomplete_window_flag"] = out["observed_minutes"] < EXPECTED_MINUTES_PER_WINDOW

        for col in AVAILABILITY_COLUMNS:
            out[col] = bool(group[col].iloc[0])

        resampled_parts.append(out.reset_index(drop=True))
        audit_rows.append({
            "step": "08_resample_15min",
            "action": "Resample each year-floor-zone group to 15-minute intervals",
            "year": year,
            "floor": floor,
            "zone": zone,
            "minute_rows_before": len(group),
            "rows_after_resample": len(out),
            "missing_time_windows": int(out["missing_time_flag"].sum()),
            "incomplete_15min_windows": int(out["incomplete_window_flag"].sum()),
            "min_timestamp": out["timestamp"].min(),
            "max_timestamp": out["timestamp"].max(),
        })

    cleaned = pd.concat(resampled_parts, ignore_index=True)
    cleaned = cleaned.sort_values(["floor", "zone", "timestamp"]).reset_index(drop=True)
    audit = pd.DataFrame(audit_rows)
    return cleaned, audit

cleaned_15min_without_measurement_flags, audit_08_resample = clean_step_08_resample_15min(zone_level_minute_sorted_df)
display(audit_08_resample.head(10))
print("15-minute rows:", f"{len(cleaned_15min_without_measurement_flags):,}")
print("Missing time windows:", int(audit_08_resample["missing_time_windows"].sum()))
print("Incomplete 15-minute windows:", int(audit_08_resample["incomplete_15min_windows"].sum()))


,step,action,year,floor,zone,minute_rows_before,rows_after_resample,missing_time_windows,incomplete_15min_windows,min_timestamp,max_timestamp
0,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,3,1,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
1,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,3,2,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
2,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,3,3,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
3,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,3,4,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
4,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,3,5,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
5,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,4,1,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
6,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,4,2,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
7,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,4,3,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
8,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,4,4,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
9,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,4,5,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00


15-minute rows: 1,054,080
Missing time windows: 0
Incomplete 15-minute windows: 0


## 19. Step 09: add measurement missing flags

What this step does:

- Adds one missing flag for each main measurement column.
- Keeps real `0` values unchanged.
- Returns missing counts and percentages for each measurement.


In [19]:
def clean_step_09_add_measurement_missing_flags(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = df.copy()
    audit_rows = []

    for col in VALUE_COLUMNS:
        flag_col = f"{col}_missing_flag"
        out[flag_col] = out[col].isna()
        audit_rows.append({
            "step": "09_add_measurement_missing_flags",
            "action": "Add missing flag for measurement column",
            "measurement_column": col,
            "flag_column": flag_col,
            "missing_count": int(out[flag_col].sum()),
            "missing_pct": float(out[flag_col].mean() * 100),
            "zero_count_preserved": int((out[col] == 0).sum(skipna=True)),
        })

    audit = pd.DataFrame(audit_rows)
    return out, audit

cleaned_15min, audit_09_missing_flags = clean_step_09_add_measurement_missing_flags(cleaned_15min_without_measurement_flags)
display(audit_09_missing_flags)


,step,action,measurement_column,flag_column,missing_count,missing_pct,zero_count_preserved
0,09_add_measurement_missing_flags,Add missing flag for measurement column,ac_power_kw,ac_power_kw_missing_flag,259233,24.593295,564649
1,09_add_measurement_missing_flags,Add missing flag for measurement column,light_power_kw,light_power_kw_missing_flag,41946,3.979394,363933
2,09_add_measurement_missing_flags,Add missing flag for measurement column,plug_power_kw,plug_power_kw_missing_flag,38413,3.644221,36102
3,09_add_measurement_missing_flags,Add missing flag for measurement column,total_power_kw,total_power_kw_missing_flag,33365,3.165320,23394
4,09_add_measurement_missing_flags,Add missing flag for measurement column,temperature_c,temperature_c_missing_flag,515539,48.908906,0
5,09_add_measurement_missing_flags,Add missing flag for measurement column,humidity_pct,humidity_pct_missing_flag,515539,48.908906,0
6,09_add_measurement_missing_flags,Add missing flag for measurement column,lux,lux_missing_flag,515543,48.909286,256692


## 20. Step 10: final validation

What this step does:

- Checks final row and column counts.
- Confirms there are no duplicate `floor + zone + timestamp` records.
- Confirms selected floors and zones.
- Summarizes all cleaning audit outputs.


In [20]:
def clean_step_10_final_validation(cleaned_df: pd.DataFrame, audit_tables: list[pd.DataFrame]) -> tuple[pd.DataFrame, pd.DataFrame]:
    duplicate_final_rows = int(cleaned_df.duplicated(["floor", "zone", "timestamp"]).sum())
    final_audit = pd.DataFrame([{
        "step": "10_final_validation",
        "action": "Validate cleaned 15-minute zone-level table",
        "rows": len(cleaned_df),
        "columns": cleaned_df.shape[1],
        "floors": ",".join(map(str, sorted(cleaned_df["floor"].unique()))),
        "zones": ",".join(map(str, sorted(cleaned_df["zone"].unique()))),
        "min_timestamp": cleaned_df["timestamp"].min(),
        "max_timestamp": cleaned_df["timestamp"].max(),
        "duplicate_floor_zone_timestamp_rows": duplicate_final_rows,
        "missing_time_windows": int(cleaned_df["missing_time_flag"].sum()),
        "incomplete_15min_windows": int(cleaned_df["incomplete_window_flag"].sum()),
    }])

    all_audits = pd.concat(audit_tables + [final_audit], ignore_index=True, sort=False)
    return final_audit, all_audits

final_validation_summary, all_cleaning_audits = clean_step_10_final_validation(
    cleaned_15min,
    [
        audit_01_timestamp,
        audit_02_sort,
        audit_03_duplicates,
        audit_04_numeric,
        audit_05_invalid_values,
        audit_06_zone_level,
        audit_07_zone_sort,
        audit_08_resample,
        audit_09_missing_flags,
    ],
)

display(final_validation_summary)


,step,action,rows,columns,floors,zones,min_timestamp,max_timestamp,duplicate_floor_zone_timestamp_rows,missing_time_windows,incomplete_15min_windows
0,10_final_validation,Validate cleaned 15-minute zone-level table,1054080,46,"3,4,5,7","1,2,3,4,5",2018-07-01,2019-12-31 23:45:00,0,0,0


## 21. Step 11: save cleaned outputs

What this step does:

- Saves the complete cleaned 15-minute table locally as `.csv.gz`.
- Saves a small preview CSV for GitHub review.
- Saves audit, invalid-value, duplicate, and missing-value summaries.


In [21]:
def clean_step_11_save_cleaning_outputs(cleaned_df: pd.DataFrame) -> pd.DataFrame:
    output_paths = {
        "cleaned_15min_file": OUTPUT_DIR / "cleaned_zone_15min_2018_2019_floors_3_4_5_7.csv.gz",
        "cleaned_preview_file": OUTPUT_DIR / "cleaned_zone_15min_2018_2019_floors_3_4_5_7_preview.csv",
        "all_cleaning_audits_file": OUTPUT_DIR / "all_cleaning_audits_2018_2019_floors_3_4_5_7.csv",
        "cleaning_summary_file": OUTPUT_DIR / "cleaning_summary_2018_2019_floors_3_4_5_7.csv",
        "duplicate_summary_file": OUTPUT_DIR / "duplicate_summary_2018_2019_floors_3_4_5_7.csv",
        "invalid_values_summary_file": OUTPUT_DIR / "invalid_values_summary_2018_2019_floors_3_4_5_7.csv",
        "missing_summary_file": OUTPUT_DIR / "missing_summary_2018_2019_floors_3_4_5_7.csv",
        "final_validation_summary_file": OUTPUT_DIR / "final_validation_summary_2018_2019_floors_3_4_5_7.csv",
    }

    cleaned_df.to_csv(output_paths["cleaned_15min_file"], index=False, compression="gzip")
    cleaned_df.head(1000).to_csv(output_paths["cleaned_preview_file"], index=False)
    all_cleaning_audits.to_csv(output_paths["all_cleaning_audits_file"], index=False)
    all_cleaning_audits.to_csv(output_paths["cleaning_summary_file"], index=False)
    audit_03_duplicates.to_csv(output_paths["duplicate_summary_file"], index=False)
    audit_05_invalid_values.to_csv(output_paths["invalid_values_summary_file"], index=False)
    audit_09_missing_flags.to_csv(output_paths["missing_summary_file"], index=False)
    final_validation_summary.to_csv(output_paths["final_validation_summary_file"], index=False)

    save_audit = pd.DataFrame([{
        "step": "11_save_cleaning_outputs",
        "action": "Save cleaned table, preview, and audit outputs",
        "output_name": name,
        "output_path": str(path),
        "file_exists": path.exists(),
        "file_size_mb": path.stat().st_size / 1024 / 1024 if path.exists() else 0,
    } for name, path in output_paths.items()])
    return save_audit

save_audit = clean_step_11_save_cleaning_outputs(cleaned_15min)
display(save_audit)
save_audit

,step,action,output_name,output_path,file_exists,file_size_mb
0,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",cleaned_15min_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,27.858608
1,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",cleaned_preview_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.283359
2,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",all_cleaning_audits_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.024652
3,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",cleaning_summary_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.024652
4,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",duplicate_summary_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.000218
5,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",invalid_values_summary_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.002723
6,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",missing_summary_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.001028
7,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",final_validation_summary_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.000269


,step,action,output_name,output_path,file_exists,file_size_mb
0,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",cleaned_15min_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,27.858608
1,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",cleaned_preview_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.283359
2,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",all_cleaning_audits_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.024652
3,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",cleaning_summary_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.024652
4,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",duplicate_summary_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.000218
5,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",invalid_values_summary_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.002723
6,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",missing_summary_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.001028
7,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",final_validation_summary_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.000269


## 22. Exploratory Data Analysis

This section uses the cleaned 15-minute zone-level table as the EDA input. Each EDA task is kept in its own cell, wrapped in a function, and returns a table that records what was checked.

In [22]:
EDA_OUTPUT_PREFIX = "2018_2019_floors_3_4_5_7"
EDA_CLEANED_FILE = OUTPUT_DIR / f"cleaned_zone_15min_{EDA_OUTPUT_PREFIX}.csv.gz"

MEASUREMENT_COLUMNS = [
    "ac_power_kw",
    "light_power_kw",
    "plug_power_kw",
    "total_power_kw",
    "temperature_c",
    "humidity_pct",
    "lux",
]

EDA_ID_COLUMNS = ["timestamp", "year", "floor", "zone"]

EDA_OUTPUT_FILES = {}


## 23. EDA Step 01: load cleaned data

What this does: reads the cleaned `.csv.gz` file, parses the `timestamp` column, and returns a loading summary with row count, column count, time range, floors, and zones.

In [23]:
def eda_step_01_load_cleaned_data(file_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not file_path.exists():
        raise FileNotFoundError(f"Cleaned file not found: {file_path}")

    df = pd.read_csv(file_path, parse_dates=["timestamp"])

    summary = pd.DataFrame([
        {
            "analysis_step": "EDA Step 01 - load cleaned data",
            "file_name": file_path.name,
            "rows": len(df),
            "columns": df.shape[1],
            "min_timestamp": df["timestamp"].min(),
            "max_timestamp": df["timestamp"].max(),
            "floors": ", ".join(map(str, sorted(df["floor"].dropna().unique()))),
            "zones": ", ".join(map(str, sorted(df["zone"].dropna().unique()))),
        }
    ])

    return df, summary


eda_df, eda_load_summary = eda_step_01_load_cleaned_data(EDA_CLEANED_FILE)
display(eda_load_summary)


,analysis_step,file_name,rows,columns,min_timestamp,max_timestamp,floors,zones
0,EDA Step 01 - load cleaned data,cleaned_zone_15min_2018_2019_floors_3_4_5_7.cs...,1054080,46,2018-07-01,2019-12-31 23:45:00,"3, 4, 5, 7","1, 2, 3, 4, 5"


## 24. EDA Step 02: inspect shape and data types

What this does: checks the table shape, memory usage, column names, and data types. It returns one overview table and one column catalog.

In [24]:
def eda_step_02_inspect_structure(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    overview = pd.DataFrame([
        {
            "analysis_step": "EDA Step 02 - inspect shape and data types",
            "rows": len(df),
            "columns": df.shape[1],
            "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
            "duplicate_rows": int(df.duplicated().sum()),
            "numeric_columns": int(df.select_dtypes(include="number").shape[1]),
            "datetime_columns": int(df.select_dtypes(include="datetime").shape[1]),
        }
    ])

    column_catalog = pd.DataFrame({
        "column_name": df.columns,
        "dtype": [str(dtype) for dtype in df.dtypes],
        "non_null_count": df.notna().sum().values,
        "null_count": df.isna().sum().values,
        "unique_count": [df[col].nunique(dropna=False) for col in df.columns],
    })

    return overview, column_catalog


eda_structure_summary, eda_column_catalog = eda_step_02_inspect_structure(eda_df)
display(eda_structure_summary)
display(eda_column_catalog)


,analysis_step,rows,columns,memory_mb,duplicate_rows,numeric_columns,datetime_columns
0,EDA Step 02 - inspect shape and data types,1054080,46,278.45,0,32,1


,column_name,dtype,non_null_count,null_count,unique_count
0,timestamp,datetime64[ns],1054080,0,52704
1,year,int64,1054080,0,2
2,floor,int64,1054080,0,4
3,zone,int64,1054080,0,5
4,ac_power_kw,float64,794847,259233,59395
5,light_power_kw,float64,1012134,41946,22017
6,plug_power_kw,float64,1015667,38413,9459
7,total_power_kw,float64,1020715,33365,80823
8,temperature_c,float64,538541,515539,48971
9,humidity_pct,float64,538541,515539,82602


## 25. EDA Step 03: check floor and zone coverage

What this does: counts records for each `floor` and `zone` combination. This confirms whether all selected floors and zones have the expected amount of data.

In [25]:
def eda_step_03_floor_zone_coverage(df: pd.DataFrame) -> pd.DataFrame:
    coverage = (
        df.groupby(["floor", "zone"], as_index=False)
        .agg(
            rows=("timestamp", "size"),
            min_timestamp=("timestamp", "min"),
            max_timestamp=("timestamp", "max"),
            missing_time_windows=("missing_time_flag", "sum"),
            incomplete_15min_windows=("incomplete_window_flag", "sum"),
        )
        .sort_values(["floor", "zone"])
        .reset_index(drop=True)
    )

    coverage["analysis_step"] = "EDA Step 03 - floor and zone coverage"
    return coverage[["analysis_step", "floor", "zone", "rows", "min_timestamp", "max_timestamp", "missing_time_windows", "incomplete_15min_windows"]]


eda_floor_zone_coverage = eda_step_03_floor_zone_coverage(eda_df)
display(eda_floor_zone_coverage)


,analysis_step,floor,zone,rows,min_timestamp,max_timestamp,missing_time_windows,incomplete_15min_windows
0,EDA Step 03 - floor and zone coverage,3,1,52704,2018-07-01,2019-12-31 23:45:00,0,0
1,EDA Step 03 - floor and zone coverage,3,2,52704,2018-07-01,2019-12-31 23:45:00,0,0
2,EDA Step 03 - floor and zone coverage,3,3,52704,2018-07-01,2019-12-31 23:45:00,0,0
3,EDA Step 03 - floor and zone coverage,3,4,52704,2018-07-01,2019-12-31 23:45:00,0,0
4,EDA Step 03 - floor and zone coverage,3,5,52704,2018-07-01,2019-12-31 23:45:00,0,0
5,EDA Step 03 - floor and zone coverage,4,1,52704,2018-07-01,2019-12-31 23:45:00,0,0
6,EDA Step 03 - floor and zone coverage,4,2,52704,2018-07-01,2019-12-31 23:45:00,0,0
7,EDA Step 03 - floor and zone coverage,4,3,52704,2018-07-01,2019-12-31 23:45:00,0,0
8,EDA Step 03 - floor and zone coverage,4,4,52704,2018-07-01,2019-12-31 23:45:00,0,0
9,EDA Step 03 - floor and zone coverage,4,5,52704,2018-07-01,2019-12-31 23:45:00,0,0


## 26. EDA Step 04: check missing values by column

What this does: counts missing values for every column and calculates the missing percentage. This helps decide which features may need imputation or missing flags.

In [26]:
def eda_step_04_missing_values_by_column(df: pd.DataFrame) -> pd.DataFrame:
    missing = pd.DataFrame({
        "column_name": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100).round(4),
        "non_missing_count": df.notna().sum().values,
    })

    missing = missing.sort_values(["missing_count", "column_name"], ascending=[False, True]).reset_index(drop=True)
    missing["analysis_step"] = "EDA Step 04 - missing values by column"
    return missing[["analysis_step", "column_name", "missing_count", "missing_pct", "non_missing_count"]]


eda_missing_by_column = eda_step_04_missing_values_by_column(eda_df)
display(eda_missing_by_column)


,analysis_step,column_name,missing_count,missing_pct,non_missing_count
0,EDA Step 04 - missing values by column,lux,515543,48.9093,538537
1,EDA Step 04 - missing values by column,lux_max,515543,48.9093,538537
2,EDA Step 04 - missing values by column,lux_min,515543,48.9093,538537
3,EDA Step 04 - missing values by column,humidity_pct,515539,48.9089,538541
4,EDA Step 04 - missing values by column,humidity_pct_max,515539,48.9089,538541
5,EDA Step 04 - missing values by column,humidity_pct_min,515539,48.9089,538541
6,EDA Step 04 - missing values by column,temperature_c,515539,48.9089,538541
7,EDA Step 04 - missing values by column,temperature_c_max,515539,48.9089,538541
8,EDA Step 04 - missing values by column,temperature_c_min,515539,48.9089,538541
9,EDA Step 04 - missing values by column,ac_power_kw,259233,24.5933,794847


## 27. EDA Step 05: check missing values by floor and zone

What this does: counts missing values for the main measurement columns within each `floor` and `zone`. This shows whether missingness is concentrated in specific building areas.

In [27]:
def eda_step_05_missing_values_by_floor_zone(df: pd.DataFrame, measurement_columns: list[str]) -> pd.DataFrame:
    available_measurements = [col for col in measurement_columns if col in df.columns]

    records = []
    for (floor, zone), group in df.groupby(["floor", "zone"], sort=True):
        for column in available_measurements:
            records.append({
                "analysis_step": "EDA Step 05 - missing values by floor and zone",
                "floor": floor,
                "zone": zone,
                "column_name": column,
                "rows": len(group),
                "missing_count": int(group[column].isna().sum()),
                "missing_pct": round(group[column].isna().mean() * 100, 4),
            })

    return pd.DataFrame(records).sort_values(["floor", "zone", "missing_count", "column_name"], ascending=[True, True, False, True]).reset_index(drop=True)


eda_missing_by_floor_zone = eda_step_05_missing_values_by_floor_zone(eda_df, MEASUREMENT_COLUMNS)
display(eda_missing_by_floor_zone)


,analysis_step,floor,zone,column_name,rows,missing_count,missing_pct
0,EDA Step 05 - missing values by floor and zone,3,1,humidity_pct,52704,19023,36.0940
1,EDA Step 05 - missing values by floor and zone,3,1,lux,52704,19023,36.0940
2,EDA Step 05 - missing values by floor and zone,3,1,temperature_c,52704,19023,36.0940
3,EDA Step 05 - missing values by floor and zone,3,1,ac_power_kw,52704,3958,7.5099
4,EDA Step 05 - missing values by floor and zone,3,1,plug_power_kw,52704,3941,7.4776
...,...,...,...,...,...,...,...
135,EDA Step 05 - missing values by floor and zone,7,5,temperature_c,52704,18332,34.7829
136,EDA Step 05 - missing values by floor and zone,7,5,ac_power_kw,52704,1986,3.7682
137,EDA Step 05 - missing values by floor and zone,7,5,plug_power_kw,52704,157,0.2979
138,EDA Step 05 - missing values by floor and zone,7,5,light_power_kw,52704,154,0.2922


## 28. EDA Step 06: summarize numeric measurement distributions

What this does: calculates count, missing count, mean, median, standard deviation, minimum, quartiles, and maximum for the main measurement columns.

In [28]:
def eda_step_06_numeric_distribution_summary(df: pd.DataFrame, measurement_columns: list[str]) -> pd.DataFrame:
    available_measurements = [col for col in measurement_columns if col in df.columns]
    summary = df[available_measurements].describe(percentiles=[0.25, 0.5, 0.75]).T.reset_index()
    summary = summary.rename(columns={
        "index": "column_name",
        "count": "non_missing_count",
        "mean": "mean",
        "std": "std",
        "min": "min",
        "25%": "q25",
        "50%": "median",
        "75%": "q75",
        "max": "max",
    })
    summary["missing_count"] = [int(df[col].isna().sum()) for col in summary["column_name"]]
    summary["missing_pct"] = [round(df[col].isna().mean() * 100, 4) for col in summary["column_name"]]
    summary["analysis_step"] = "EDA Step 06 - numeric measurement distributions"

    ordered_columns = ["analysis_step", "column_name", "non_missing_count", "missing_count", "missing_pct", "mean", "std", "min", "q25", "median", "q75", "max"]
    return summary[ordered_columns].round(4)


eda_numeric_summary = eda_step_06_numeric_distribution_summary(eda_df, MEASUREMENT_COLUMNS)
display(eda_numeric_summary)


,analysis_step,column_name,non_missing_count,missing_count,missing_pct,mean,std,min,q25,median,q75,max
0,EDA Step 06 - numeric measurement distributions,ac_power_kw,794847.0,259233,24.5933,4.0986,8.7577,0.0000,0.0000,0.0000,0.0700,50.7650
1,EDA Step 06 - numeric measurement distributions,light_power_kw,1012134.0,41946,3.9794,1.1127,1.6088,0.0000,0.0000,0.0900,2.0960,78.5544
2,EDA Step 06 - numeric measurement distributions,plug_power_kw,1015667.0,38413,3.6442,0.3041,0.4282,0.0000,0.0500,0.1800,0.3847,33.9400
3,EDA Step 06 - numeric measurement distributions,total_power_kw,1020715.0,33365,3.1653,4.5924,9.1445,0.0000,0.0780,0.3913,3.1080,51.6093
4,EDA Step 06 - numeric measurement distributions,temperature_c,538541.0,515539,48.9089,24.9956,2.8746,10.9133,23.6280,25.3673,26.9200,35.0460
5,EDA Step 06 - numeric measurement distributions,humidity_pct,538541.0,515539,48.9089,65.9087,6.4217,37.3793,61.5536,66.1840,70.4987,88.8473
6,EDA Step 06 - numeric measurement distributions,lux,538537.0,515543,48.9093,17.3355,24.2623,0.0000,0.0000,0.9333,37.0000,101.8000


## 29. EDA Step 07: compare measurements by floor

What this does: calculates average, median, and maximum values by floor for the main measurement columns. This helps compare energy and environmental patterns across floors.

In [29]:
def eda_step_07_measurements_by_floor(df: pd.DataFrame, measurement_columns: list[str]) -> pd.DataFrame:
    available_measurements = [col for col in measurement_columns if col in df.columns]
    by_floor = (
        df.groupby("floor")[available_measurements]
        .agg(["mean", "median", "max"])
        .round(4)
    )
    by_floor.columns = [f"{column}_{stat}" for column, stat in by_floor.columns]
    by_floor = by_floor.reset_index()
    by_floor.insert(0, "analysis_step", "EDA Step 07 - measurements by floor")
    return by_floor


eda_measurements_by_floor = eda_step_07_measurements_by_floor(eda_df, MEASUREMENT_COLUMNS)
display(eda_measurements_by_floor)


,analysis_step,floor,ac_power_kw_mean,ac_power_kw_median,ac_power_kw_max,light_power_kw_mean,light_power_kw_median,light_power_kw_max,plug_power_kw_mean,plug_power_kw_median,plug_power_kw_max,total_power_kw_mean,total_power_kw_median,total_power_kw_max,temperature_c_mean,temperature_c_median,temperature_c_max,humidity_pct_mean,humidity_pct_median,humidity_pct_max,lux_mean,lux_median,lux_max
0,EDA Step 07 - measurements by floor,3,3.4882,0.0,40.0517,1.2303,0.0400,78.5544,0.2889,0.1153,10.5610,4.1806,0.4260,51.6093,24.6433,25.1340,30.8000,65.0663,64.8289,88.5353,16.4016,0.0000,83.5333
1,EDA Step 07 - measurements by floor,4,3.5885,0.0,50.7650,0.9267,0.1000,10.0420,0.4490,0.3021,10.4120,4.2003,0.4467,50.7650,24.8768,25.2433,31.6340,65.0057,64.7733,83.1620,20.9170,0.2667,101.8000
2,EDA Step 07 - measurements by floor,5,3.5157,0.0,37.1193,1.1495,0.0673,9.4507,0.3449,0.2560,33.9400,4.2036,0.3600,45.0467,24.6269,25.1557,30.7733,67.9444,68.2037,88.8473,15.6977,1.0000,78.3333
3,EDA Step 07 - measurements by floor,7,5.7313,0.0,39.3600,1.1412,0.1440,5.7160,0.1407,0.0773,1.6793,5.7354,0.2853,42.8887,25.7790,26.3753,35.0460,65.7318,66.4153,87.3420,16.2573,0.9333,93.6000


## 30. EDA Step 08: compare measurements by floor and zone

What this does: calculates average values for each `floor` and `zone`. This identifies which floor-zone combinations have higher power use or different environmental conditions.

In [30]:
def eda_step_08_measurements_by_floor_zone(df: pd.DataFrame, measurement_columns: list[str]) -> pd.DataFrame:
    available_measurements = [col for col in measurement_columns if col in df.columns]
    by_floor_zone = (
        df.groupby(["floor", "zone"], as_index=False)[available_measurements]
        .mean()
        .round(4)
        .sort_values(["floor", "zone"])
        .reset_index(drop=True)
    )
    by_floor_zone.insert(0, "analysis_step", "EDA Step 08 - measurements by floor and zone")
    return by_floor_zone


eda_measurements_by_floor_zone = eda_step_08_measurements_by_floor_zone(eda_df, MEASUREMENT_COLUMNS)
display(eda_measurements_by_floor_zone)


,analysis_step,floor,zone,ac_power_kw,light_power_kw,plug_power_kw,total_power_kw,temperature_c,humidity_pct,lux
0,EDA Step 08 - measurements by floor and zone,3,1,4.4934,1.7606,0.2518,6.5012,25.6704,61.2418,12.8147
1,EDA Step 08 - measurements by floor and zone,3,2,3.6897,2.1690,0.3120,5.9433,25.2122,61.1289,29.4543
2,EDA Step 08 - measurements by floor and zone,3,3,NaN,0.4463,0.4489,0.8950,NaN,NaN,NaN
3,EDA Step 08 - measurements by floor and zone,3,4,3.8720,1.0271,0.2196,4.8818,23.8958,67.6038,10.3913
4,EDA Step 08 - measurements by floor and zone,3,5,1.8349,0.7478,0.2121,2.6793,23.7100,70.9378,12.3398
5,EDA Step 08 - measurements by floor and zone,4,1,4.7285,1.5848,1.0075,7.2160,25.6875,62.4221,23.1904
6,EDA Step 08 - measurements by floor and zone,4,2,3.3887,0.7417,0.2250,4.2705,24.6451,66.4044,17.3051
7,EDA Step 08 - measurements by floor and zone,4,3,NaN,0.3479,0.3557,0.7035,NaN,NaN,NaN
8,EDA Step 08 - measurements by floor and zone,4,4,4.4852,1.4967,0.4074,6.3532,24.6304,63.3787,25.2943
9,EDA Step 08 - measurements by floor and zone,4,5,1.7440,0.4926,0.2350,2.4575,24.4974,67.5879,18.5299


## 31. EDA Step 09: analyze hourly patterns

What this does: extracts hour from `timestamp` and calculates the hourly average of the main measurement columns. This shows daily usage patterns.

In [31]:
def eda_step_09_hourly_patterns(df: pd.DataFrame, measurement_columns: list[str]) -> pd.DataFrame:
    available_measurements = [col for col in measurement_columns if col in df.columns]
    hourly = df.assign(hour=df["timestamp"].dt.hour)
    hourly_summary = (
        hourly.groupby("hour", as_index=False)[available_measurements]
        .mean()
        .round(4)
        .sort_values("hour")
        .reset_index(drop=True)
    )
    hourly_summary.insert(0, "analysis_step", "EDA Step 09 - hourly patterns")
    return hourly_summary


eda_hourly_patterns = eda_step_09_hourly_patterns(eda_df, MEASUREMENT_COLUMNS)
display(eda_hourly_patterns)


,analysis_step,hour,ac_power_kw,light_power_kw,plug_power_kw,total_power_kw,temperature_c,humidity_pct,lux
0,EDA Step 09 - hourly patterns,0,0.0113,0.0260,0.1331,0.1671,26.0302,66.9966,0.1164
1,EDA Step 09 - hourly patterns,1,0.0112,0.0249,0.1330,0.1658,26.0239,67.2172,0.1154
2,EDA Step 09 - hourly patterns,2,0.0111,0.0241,0.1325,0.1644,26.0282,67.4153,0.1136
3,EDA Step 09 - hourly patterns,3,0.0110,0.0236,0.1325,0.1639,26.0221,67.5997,0.1197
4,EDA Step 09 - hourly patterns,4,0.0105,0.0284,0.1325,0.1682,26.0219,67.7594,0.1316
5,EDA Step 09 - hourly patterns,5,0.0040,0.1631,0.1380,0.3019,26.0172,67.9557,1.1805
6,EDA Step 09 - hourly patterns,6,0.0056,0.9949,0.2332,1.2207,26.0649,68.8187,12.5374
7,EDA Step 09 - hourly patterns,7,3.0474,1.4381,0.3176,4.1109,26.1855,69.1924,21.7296
8,EDA Step 09 - hourly patterns,8,14.3822,1.9323,0.4712,13.5776,23.1678,65.5458,32.5883
9,EDA Step 09 - hourly patterns,9,12.6647,2.0667,0.5201,12.4165,22.6071,65.3981,34.9700


## 32. EDA Step 10: analyze weekday and weekend patterns

What this does: compares weekday, day name, and weekend averages for the main measurement columns. This helps identify occupancy-related energy patterns.

In [32]:
def eda_step_10_weekday_weekend_patterns(df: pd.DataFrame, measurement_columns: list[str]) -> pd.DataFrame:
    available_measurements = [col for col in measurement_columns if col in df.columns]
    temp = df.assign(
        day_of_week=df["timestamp"].dt.dayofweek,
        day_name=df["timestamp"].dt.day_name(),
        is_weekend=df["timestamp"].dt.dayofweek >= 5,
    )

    weekday_summary = (
        temp.groupby(["day_of_week", "day_name", "is_weekend"], as_index=False)[available_measurements]
        .mean()
        .round(4)
        .sort_values("day_of_week")
        .reset_index(drop=True)
    )
    weekday_summary.insert(0, "analysis_step", "EDA Step 10 - weekday and weekend patterns")
    return weekday_summary


eda_weekday_weekend_patterns = eda_step_10_weekday_weekend_patterns(eda_df, MEASUREMENT_COLUMNS)
display(eda_weekday_weekend_patterns)


,analysis_step,day_of_week,day_name,is_weekend,ac_power_kw,light_power_kw,plug_power_kw,total_power_kw,temperature_c,humidity_pct,lux
0,EDA Step 10 - weekday and weekend patterns,0,Monday,False,5.2862,1.2905,0.3408,5.7148,25.3828,65.9062,19.8923
1,EDA Step 10 - weekday and weekend patterns,1,Tuesday,False,5.6644,1.4360,0.3727,6.1989,24.6184,64.6222,23.0952
2,EDA Step 10 - weekday and weekend patterns,2,Wednesday,False,5.6915,1.5197,0.3864,6.3250,24.2355,64.9309,23.9264
3,EDA Step 10 - weekday and weekend patterns,3,Thursday,False,5.7184,1.5168,0.3790,6.3271,24.0027,65.1777,23.7060
4,EDA Step 10 - weekday and weekend patterns,4,Friday,False,5.5734,1.4827,0.3662,6.1644,23.9046,65.5437,23.2594
5,EDA Step 10 - weekday and weekend patterns,5,Saturday,True,0.6277,0.3623,0.1500,0.9961,25.8281,66.8541,5.1256
6,EDA Step 10 - weekday and weekend patterns,6,Sunday,True,0.1515,0.1844,0.1344,0.4345,26.9693,68.3180,2.3738


## 33. EDA Step 11: analyze monthly trends

What this does: aggregates data by month and calculates monthly average measurements. This is useful for seasonal trend checks.

In [33]:
def eda_step_11_monthly_trends(df: pd.DataFrame, measurement_columns: list[str]) -> pd.DataFrame:
    available_measurements = [col for col in measurement_columns if col in df.columns]
    monthly = df.assign(month=df["timestamp"].dt.to_period("M").astype(str))
    monthly_summary = (
        monthly.groupby("month", as_index=False)[available_measurements]
        .mean()
        .round(4)
        .sort_values("month")
        .reset_index(drop=True)
    )
    monthly_summary.insert(0, "analysis_step", "EDA Step 11 - monthly trends")
    return monthly_summary


eda_monthly_trends = eda_step_11_monthly_trends(eda_df, MEASUREMENT_COLUMNS)
display(eda_monthly_trends)


,analysis_step,month,ac_power_kw,light_power_kw,plug_power_kw,total_power_kw,temperature_c,humidity_pct,lux
0,EDA Step 11 - monthly trends,2018-07,4.0079,1.0756,0.3105,4.5902,26.7828,62.7312,15.7925
1,EDA Step 11 - monthly trends,2018-08,4.1366,1.2128,0.3269,4.8463,26.6955,62.8298,18.0594
2,EDA Step 11 - monthly trends,2018-09,4.0142,1.2241,0.3260,4.7559,26.5857,62.8488,18.2584
3,EDA Step 11 - monthly trends,2018-10,4.2080,1.1179,0.3168,4.7678,NaN,NaN,NaN
4,EDA Step 11 - monthly trends,2018-11,4.5914,1.1839,0.2620,5.0077,NaN,NaN,NaN
5,EDA Step 11 - monthly trends,2018-12,3.3315,0.9285,0.2699,3.8335,NaN,NaN,NaN
6,EDA Step 11 - monthly trends,2019-01,3.3924,1.1301,0.3009,3.7058,NaN,NaN,NaN
7,EDA Step 11 - monthly trends,2019-02,3.7057,1.0782,0.2838,3.6078,NaN,NaN,NaN
8,EDA Step 11 - monthly trends,2019-03,4.2517,1.1121,0.2727,4.6458,24.6467,69.3670,17.2907
9,EDA Step 11 - monthly trends,2019-04,4.5886,0.9936,0.2539,4.7648,25.9460,68.9710,15.3053


## 34. EDA Step 12: calculate measurement correlations

What this does: calculates pairwise correlations among the main measurement columns. This helps identify relationships between energy usage and environmental variables.

In [34]:
def eda_step_12_measurement_correlations(df: pd.DataFrame, measurement_columns: list[str]) -> pd.DataFrame:
    available_measurements = [col for col in measurement_columns if col in df.columns]
    corr = df[available_measurements].corr().round(4)
    corr_long = corr.reset_index().melt(id_vars="index", var_name="column_b", value_name="correlation")
    corr_long = corr_long.rename(columns={"index": "column_a"})
    corr_long.insert(0, "analysis_step", "EDA Step 12 - measurement correlations")
    return corr_long


eda_measurement_correlations = eda_step_12_measurement_correlations(eda_df, MEASUREMENT_COLUMNS)
display(eda_measurement_correlations)


,analysis_step,column_a,column_b,correlation
0,EDA Step 12 - measurement correlations,ac_power_kw,ac_power_kw,1.0000
1,EDA Step 12 - measurement correlations,light_power_kw,ac_power_kw,0.5627
2,EDA Step 12 - measurement correlations,plug_power_kw,ac_power_kw,0.4846
3,EDA Step 12 - measurement correlations,total_power_kw,ac_power_kw,0.9863
4,EDA Step 12 - measurement correlations,temperature_c,ac_power_kw,-0.6692
5,EDA Step 12 - measurement correlations,humidity_pct,ac_power_kw,-0.1065
6,EDA Step 12 - measurement correlations,lux,ac_power_kw,0.5231
7,EDA Step 12 - measurement correlations,ac_power_kw,light_power_kw,0.5627
8,EDA Step 12 - measurement correlations,light_power_kw,light_power_kw,1.0000
9,EDA Step 12 - measurement correlations,plug_power_kw,light_power_kw,0.5781


## 35. EDA Step 13: detect potential outliers with IQR

What this does: uses the IQR rule to count potential outliers for each main measurement column. It returns lower/upper bounds and how many values fall outside those bounds.

In [35]:
def eda_step_13_iqr_outlier_summary(df: pd.DataFrame, measurement_columns: list[str]) -> pd.DataFrame:
    available_measurements = [col for col in measurement_columns if col in df.columns]
    records = []

    for column in available_measurements:
        values = df[column].dropna()
        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outlier_count = int(((values < lower_bound) | (values > upper_bound)).sum())

        records.append({
            "analysis_step": "EDA Step 13 - IQR potential outliers",
            "column_name": column,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "non_missing_count": int(values.shape[0]),
            "outlier_count": outlier_count,
            "outlier_pct": round(outlier_count / values.shape[0] * 100, 4) if values.shape[0] else 0,
        })

    return pd.DataFrame(records).round(4)


eda_iqr_outlier_summary = eda_step_13_iqr_outlier_summary(eda_df, MEASUREMENT_COLUMNS)
display(eda_iqr_outlier_summary)


,analysis_step,column_name,q1,q3,iqr,lower_bound,upper_bound,non_missing_count,outlier_count,outlier_pct
0,EDA Step 13 - IQR potential outliers,ac_power_kw,0.0000,0.0700,0.0700,-0.1050,0.1750,794847,184450,23.2057
1,EDA Step 13 - IQR potential outliers,light_power_kw,0.0000,2.0960,2.0960,-3.1440,5.2400,1012134,22597,2.2326
2,EDA Step 13 - IQR potential outliers,plug_power_kw,0.0500,0.3847,0.3347,-0.4520,0.8867,1015667,69012,6.7947
3,EDA Step 13 - IQR potential outliers,total_power_kw,0.0780,3.1080,3.0300,-4.4670,7.6530,1020715,170049,16.6598
4,EDA Step 13 - IQR potential outliers,temperature_c,23.6280,26.9200,3.2920,18.6900,31.8580,538541,16115,2.9923
5,EDA Step 13 - IQR potential outliers,humidity_pct,61.5536,70.4987,8.9451,48.1359,83.9163,538541,3057,0.5676
6,EDA Step 13 - IQR potential outliers,lux,0.0000,37.0000,37.0000,-55.5000,92.5000,538537,130,0.0241


## 36. EDA Step 14: inspect highest total power records

What this does: returns the top records with the highest `total_power_kw`. This helps manually inspect peak-load periods before feature engineering.

In [36]:
def eda_step_14_highest_total_power_records(df: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    required_columns = ["timestamp", "year", "floor", "zone", "ac_power_kw", "light_power_kw", "plug_power_kw", "total_power_kw", "temperature_c", "humidity_pct", "lux"]
    available_columns = [col for col in required_columns if col in df.columns]

    top_records = (
        df[available_columns]
        .sort_values("total_power_kw", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    top_records.insert(0, "analysis_step", "EDA Step 14 - highest total power records")
    return top_records


eda_highest_total_power_records = eda_step_14_highest_total_power_records(eda_df, top_n=20)
display(eda_highest_total_power_records)


,analysis_step,timestamp,year,floor,zone,ac_power_kw,light_power_kw,plug_power_kw,total_power_kw,temperature_c,humidity_pct,lux
0,EDA Step 14 - highest total power records,2018-10-11 10:30:00,2018,3,2,NaN,78.554444,4.476667,51.609333,NaN,NaN,NaN
1,EDA Step 14 - highest total power records,2018-09-19 13:30:00,2018,4,4,50.765000,NaN,NaN,50.765000,NaN,NaN,NaN
2,EDA Step 14 - highest total power records,2019-02-08 14:30:00,2019,3,4,NaN,54.564286,10.561000,48.756000,NaN,NaN,NaN
3,EDA Step 14 - highest total power records,2018-09-19 13:45:00,2018,4,1,47.755000,NaN,NaN,47.755000,NaN,NaN,NaN
4,EDA Step 14 - highest total power records,2019-05-21 13:30:00,2019,3,1,39.513333,4.875333,0.752667,45.141333,22.987333,58.379333,34.666667
5,EDA Step 14 - highest total power records,2018-07-23 10:30:00,2018,5,1,33.286000,9.450667,2.310000,45.046667,16.204000,73.161333,55.800000
6,EDA Step 14 - highest total power records,2019-06-04 14:00:00,2019,4,1,37.188000,4.536667,3.217333,44.942000,20.686667,62.511333,58.933333
7,EDA Step 14 - highest total power records,2019-05-21 13:15:00,2019,3,1,39.564000,4.706000,0.636000,44.906000,24.484000,58.440000,36.200000
8,EDA Step 14 - highest total power records,2019-05-27 13:15:00,2019,4,1,37.020000,4.670667,3.196000,44.886667,22.558000,60.967333,67.733333
9,EDA Step 14 - highest total power records,2019-07-23 13:15:00,2019,3,1,39.391333,5.003333,0.484000,44.878667,24.050000,57.386667,33.933333


## 37. EDA Step 15: save EDA output tables

What this does: saves the EDA summary tables to the `outputs/` folder so they can be reused in the report without rerunning every cell.

In [37]:
def eda_step_15_save_eda_outputs(output_dir: Path, prefix: str, tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    saved_records = []

    for table_name, table in tables.items():
        file_path = output_dir / f"eda_{table_name}_{prefix}.csv"
        table.to_csv(file_path, index=False)
        saved_records.append({
            "analysis_step": "EDA Step 15 - save EDA output tables",
            "table_name": table_name,
            "rows_saved": len(table),
            "columns_saved": table.shape[1],
            "file_path": str(file_path),
        })
        EDA_OUTPUT_FILES[table_name] = file_path

    return pd.DataFrame(saved_records)


eda_output_tables = {
    "load_summary": eda_load_summary,
    "structure_summary": eda_structure_summary,
    "column_catalog": eda_column_catalog,
    "floor_zone_coverage": eda_floor_zone_coverage,
    "missing_by_column": eda_missing_by_column,
    "missing_by_floor_zone": eda_missing_by_floor_zone,
    "numeric_summary": eda_numeric_summary,
    "measurements_by_floor": eda_measurements_by_floor,
    "measurements_by_floor_zone": eda_measurements_by_floor_zone,
    "hourly_patterns": eda_hourly_patterns,
    "weekday_weekend_patterns": eda_weekday_weekend_patterns,
    "monthly_trends": eda_monthly_trends,
    "measurement_correlations": eda_measurement_correlations,
    "iqr_outlier_summary": eda_iqr_outlier_summary,
    "highest_total_power_records": eda_highest_total_power_records,
}

eda_save_summary = eda_step_15_save_eda_outputs(OUTPUT_DIR, EDA_OUTPUT_PREFIX, eda_output_tables)
display(eda_save_summary)


,analysis_step,table_name,rows_saved,columns_saved,file_path
0,EDA Step 15 - save EDA output tables,load_summary,1,8,/Users/yonglijiang/Documents/Adelaide Universi...
1,EDA Step 15 - save EDA output tables,structure_summary,1,7,/Users/yonglijiang/Documents/Adelaide Universi...
2,EDA Step 15 - save EDA output tables,column_catalog,46,5,/Users/yonglijiang/Documents/Adelaide Universi...
3,EDA Step 15 - save EDA output tables,floor_zone_coverage,20,8,/Users/yonglijiang/Documents/Adelaide Universi...
4,EDA Step 15 - save EDA output tables,missing_by_column,46,5,/Users/yonglijiang/Documents/Adelaide Universi...
5,EDA Step 15 - save EDA output tables,missing_by_floor_zone,140,7,/Users/yonglijiang/Documents/Adelaide Universi...
6,EDA Step 15 - save EDA output tables,numeric_summary,7,12,/Users/yonglijiang/Documents/Adelaide Universi...
7,EDA Step 15 - save EDA output tables,measurements_by_floor,4,23,/Users/yonglijiang/Documents/Adelaide Universi...
8,EDA Step 15 - save EDA output tables,measurements_by_floor_zone,20,10,/Users/yonglijiang/Documents/Adelaide Universi...
9,EDA Step 15 - save EDA output tables,hourly_patterns,24,9,/Users/yonglijiang/Documents/Adelaide Universi...
